In [13]:
import os

os.environ["S3_ACCESS_KEY"] = "user_37sKcYrvnk9UXaIY3B3Zr90MH0g"
os.environ["S3_SECRET_KEY"] = "rps_YW72UMRXEMRVC8A407OCL08J8G34U1B3QTNO1ETX18pa1n"
os.environ["S3_BUCKET"] = "e9tcw5eupu"
os.environ["S3_ENDPOINT_URL"] = "https://s3api-eu-ro-1.runpod.io"
os.environ["S3_REGION"] = "eu-ro-1"

In [14]:
import os
import sys
import subprocess
from pathlib import Path
import importlib.util
import boto3

print("Starting SAM 3D Body pipeline (S3-only, repo-native loader)...")

# ---------------------------
# S3 CONFIG
# ---------------------------
# Set these before running, or replace with literals if needed.
ACCESS_KEY = os.environ.get("S3_ACCESS_KEY")
SECRET_KEY = os.environ.get("S3_SECRET_KEY")
BUCKET = os.environ.get("S3_BUCKET", "e9tcw5eupu")
ENDPOINT_URL = os.environ.get("S3_ENDPOINT_URL", "https://s3api-eu-ro-1.runpod.io")
REGION = os.environ.get("S3_REGION", "eu-ro-1")

if not ACCESS_KEY or not SECRET_KEY:
    raise RuntimeError("Missing S3_ACCESS_KEY / S3_SECRET_KEY environment variables.")

s3 = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    endpoint_url=ENDPOINT_URL,
    region_name=REGION,
)

# ---------------------------
# PATHS
# ---------------------------
WORKSPACE_DIR = Path.cwd()
CACHE_DIR = Path("/workspace/cache")

REPO_PREFIX = "sam-3d-body"
MODEL_PREFIX = "sam-3d-body-model"

REPO_DIR = CACHE_DIR / "sam-3d-body"
MODEL_DIR = CACHE_DIR / "sam-3d-body-model"

INPUT_IMAGE_PATH = WORKSPACE_DIR / "man.png"
OUTPUT_GLB_PATH = WORKSPACE_DIR / "man_body_mesh.glb"

MODEL_CKPT_PATH = MODEL_DIR / "model.ckpt"
MODEL_CFG_PATH = MODEL_DIR / "model_config.yaml"
MHR_MODEL_PATH = MODEL_DIR / "assets" / "mhr_model.pt"

# ---------------------------
# HELPERS
# ---------------------------
def pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", *packages]
    print("Installing:", " ".join(packages))
    subprocess.check_call(cmd)

def pip_uninstall(*packages):
    cmd = [sys.executable, "-m", "pip", "uninstall", "-y", *packages]
    print("Uninstalling:", " ".join(packages))
    subprocess.call(cmd)

def module_exists(module_name):
    return importlib.util.find_spec(module_name) is not None

def version_tuple(s):
    out = []
    for part in s.split("."):
        num = ""
        for ch in part:
            if ch.isdigit():
                num += ch
            else:
                break
        out.append(int(num) if num else 0)
    while len(out) < 3:
        out.append(0)
    return tuple(out[:3])

def get_installed_version(dist_name):
    try:
        from importlib.metadata import version
        return version(dist_name)
    except Exception:
        return None

def ensure_dirs():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------
# S3 HELPERS
# ---------------------------
def list_s3_objects(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")
    objects = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            objects.append(obj)
    return objects

def should_skip_s3_key(key):
    parts = key.split("/")
    if ".git" in parts:
        return True
    if "__pycache__" in parts:
        return True
    if key.endswith(".pyc"):
        return True
    return False

def download_s3_file(bucket, key, local_path):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading s3://{bucket}/{key} -> {local_path}")
    s3.download_file(bucket, key, str(local_path))

def sync_prefix_from_bucket(bucket, prefix, local_dir):
    print(f"Syncing prefix: {prefix}")
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    objects = list_s3_objects(bucket, prefix)
    if not objects:
        raise FileNotFoundError(f"No files found under s3://{bucket}/{prefix}")

    checked = 0
    downloaded = 0
    skipped = 0

    for obj in objects:
        key = obj["Key"]
        size = obj["Size"]

        if should_skip_s3_key(key):
            skipped += 1
            continue

        rel = key[len(prefix):].lstrip("/")
        if not rel:
            continue

        local_path = local_dir / rel
        checked += 1

        if local_path.exists() and local_path.stat().st_size == size:
            continue

        download_s3_file(bucket, key, local_path)
        downloaded += 1

    print(f"Sync done for {prefix}. checked={checked}, downloaded={downloaded}, skipped={skipped}")

# ---------------------------
# DEPENDENCIES
# ---------------------------
def bootstrap_environment():
    print("Checking environment...")

    changed_core = False
    numpy_ver = get_installed_version("numpy")
    cv2_ok = module_exists("cv2")

    if numpy_ver is None or version_tuple(numpy_ver) >= (2, 0, 0):
        print(f"Detected numpy version {numpy_ver!r}; resetting to numpy<2.")
        pip_uninstall("opencv-python", "opencv-python-headless", "opencv-contrib-python", "numpy")
        pip_install("numpy<2", "opencv-python-headless<4.11")
        changed_core = True
    elif not cv2_ok:
        print("OpenCV missing; installing opencv-python-headless.")
        pip_install("opencv-python-headless<4.11")
        changed_core = True

    if changed_core:
        raise SystemExit("Core packages changed. Restart the runtime once, then rerun this script.")

    if get_installed_version("torch") is None:
        pip_install("torch", "torchvision", "torchaudio")

    deps = [
        "pytorch-lightning",
        "yacs",
        "scikit-image",
        "einops",
        "timm",
        "dill",
        "pandas<3",
        "rich",
        "hydra-core",
        "hydra-submitit-launcher",
        "hydra-colorlog",
        "pyrootutils",
        "webdataset",
        "chump",
        "networkx==3.2.1",
        "roma",
        "joblib",
        "wandb",
        "appdirs",
        "ffmpeg-python",
        "cython",
        "jsonlines",
        "pytest",
        "xtcocotools",
        "loguru",
        "optree",
        "fvcore",
        "pycocotools",
        "trimesh",
        "Pillow",
        "tqdm",
        "omegaconf",
    ]

    for dep in deps:
        try:
            pip_install(dep)
        except subprocess.CalledProcessError as e:
            print(f"Warning: failed to install {dep}: {e}")

    print("Dependency setup complete.")

# ---------------------------
# IMPORTS
# ---------------------------
def add_repo_to_path():
    repo_root = str(REPO_DIR.resolve())
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print("Added repo to sys.path:", repo_root)

def import_runtime():
    print("Importing runtime packages...")
    import cv2
    import numpy as np
    import torch
    import trimesh

    from sam_3d_body.build_models import load_sam_3d_body
    from sam_3d_body.sam_3d_body_estimator import SAM3DBodyEstimator

    return cv2, np, torch, trimesh, load_sam_3d_body, SAM3DBodyEstimator

# ---------------------------
# VALIDATION
# ---------------------------
def validate_inputs():
    if not INPUT_IMAGE_PATH.exists():
        raise FileNotFoundError(f"Input image not found: {INPUT_IMAGE_PATH}")

    if not MODEL_CKPT_PATH.exists():
        raise FileNotFoundError(f"Missing checkpoint: {MODEL_CKPT_PATH}")

    if not MODEL_CFG_PATH.exists():
        print("Warning: Missing model config:", MODEL_CFG_PATH)

    if not MHR_MODEL_PATH.exists():
        print("Warning: Missing MHR model:", MHR_MODEL_PATH)

    print("Input and model files validated.")

# ---------------------------
# OUTPUT HELPERS
# ---------------------------
def select_output(outputs):
    print("Parsing outputs...")

    if isinstance(outputs, list):
        if len(outputs) == 0:
            raise ValueError("No person detected in the image.")
        return outputs[0]

    if isinstance(outputs, dict):
        if "pred_vertices" in outputs:
            return outputs
        if "outputs" in outputs and isinstance(outputs["outputs"], list) and outputs["outputs"]:
            return outputs["outputs"][0]
        if "predictions" in outputs and isinstance(outputs["predictions"], list) and outputs["predictions"]:
            return outputs["predictions"][0]
        raise ValueError(f"Unexpected output structure: {list(outputs.keys())}")

    raise TypeError(f"Unsupported outputs type: {type(outputs)}")

def to_numpy(x, np):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x)

def find_faces(estimator, model, outputs):
    for obj_name, obj in [
        ("estimator", estimator),
        ("model", model),
        ("outputs", outputs),
    ]:
        if hasattr(obj, "faces"):
            return getattr(obj, "faces")

    if isinstance(outputs, dict):
        for key in ["faces", "pred_faces", "mesh_faces"]:
            if key in outputs:
                return outputs[key]

    raise AttributeError("Could not find mesh faces on estimator, model, or outputs.")

# ---------------------------
# MAIN
# ---------------------------
def main():
    ensure_dirs()

    sync_prefix_from_bucket(BUCKET, REPO_PREFIX, REPO_DIR)
    sync_prefix_from_bucket(BUCKET, MODEL_PREFIX, MODEL_DIR)

    bootstrap_environment()

    add_repo_to_path()
    cv2, np, torch, trimesh, load_sam_3d_body, SAM3DBodyEstimator = import_runtime()

    validate_inputs()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    print("Loading model with repo-native loader...")
    model, model_cfg = load_sam_3d_body(
        str(MODEL_CKPT_PATH),
        device=device,
        mhr_path=str(MHR_MODEL_PATH) if MHR_MODEL_PATH.exists() else None,
    )
    model.eval()
    print("Model loaded.")

    print("Creating estimator...")
    estimator = SAM3DBodyEstimator(
        sam_3d_body_model=model,
        model_cfg=model_cfg,
        human_detector=None,
        human_segmentor=None,
        fov_estimator=None,
    )

    print("Loading input image...")
    img_bgr = cv2.imread(str(INPUT_IMAGE_PATH))
    if img_bgr is None:
        raise ValueError(f"Failed to load image: {INPUT_IMAGE_PATH}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    print("Image shape:", img_rgb.shape)

    print("Running inference...")
    outputs = estimator.process_one_image(img_rgb)
    print("Inference complete.")

    selected_output = select_output(outputs)

    if "pred_vertices" not in selected_output:
        raise KeyError("Selected output does not contain 'pred_vertices'")

    vertices = to_numpy(selected_output["pred_vertices"], np)
    faces = to_numpy(find_faces(estimator, model, selected_output), np)

    print("Raw vertices shape:", vertices.shape)
    print("Raw faces shape:", faces.shape)

    if vertices.ndim == 3 and vertices.shape[0] == 1:
        vertices = vertices[0]
    if faces.ndim == 3 and faces.shape[0] == 1:
        faces = faces[0]

    if vertices.ndim != 2 or vertices.shape[1] != 3:
        raise ValueError(f"Expected vertices shape (N, 3), got {vertices.shape}")
    if faces.ndim != 2 or faces.shape[1] != 3:
        raise ValueError(f"Expected faces shape (F, 3), got {faces.shape}")

    vertices = vertices.astype(np.float32)
    faces = faces.astype(np.int64)

    print("Creating mesh...")
    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)

    print("Exporting GLB...")
    mesh.export(str(OUTPUT_GLB_PATH))

    if not OUTPUT_GLB_PATH.exists():
        raise FileNotFoundError("GLB export failed; output file not found.")

    print("Done.")
    print("Saved GLB:", OUTPUT_GLB_PATH.resolve())
    print("Size:", OUTPUT_GLB_PATH.stat().st_size, "bytes")

if __name__ == "__main__":
    main()

Starting SAM 3D Body pipeline (S3-only, repo-native loader)...
Syncing prefix: sam-3d-body
Sync done for sam-3d-body. checked=108, downloaded=0, skipped=65
Syncing prefix: sam-3d-body-model
Sync done for sam-3d-body-model. checked=3, downloaded=0, skipped=0
Checking environment...
Installing: pytorch-lightning



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


Installing: yacs



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


Installing: scikit-image


...

Installing: omegaconf



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov3_main
Ignored kwargs: {'drop_path': 0.1}


Dependency setup complete.
Added repo to sys.path: /workspace/cache/sam-3d-body
Importing runtime packages...
Input and model files validated.
Using device: cuda
Loading model with repo-native loader...
Loading SAM 3D Body model...


The model and loaded state dict do not match exactly

missing keys in source state_dict: backbone.encoder.mask_token, head_pose.hand_pose_comps_ori, head_pose.mhr.face_expressions_model.shape_vectors, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_indices, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_weight, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.2.weight, head_pose.mhr.character_torch.skeleton.joint_translation_offsets, head_pose.mhr.character_torch.skeleton.joint_prerotations, head_pose.mhr.character_torch.skeleton.pmi, head_pose.mhr.character_torch.skeleton.joint_parents, head_pose.mhr.character_torch.mesh.rest_vertices, head_pose.mhr.character_torch.mesh.faces, head_pose.mhr.character_torch.mesh.texcoords, head_pose.mhr.character_torch.mesh.texcoord_faces, head_pose.mhr.character_torch.parameter_transform.parameter_transform, head_pose.mhr.character_torch.parameter_transform.pose_parameters, head_pose.mhr.character_torch.par

Model loaded.
Creating estimator...
No human detector is used...
Mask-condition inference is not supported...
No FOV estimator... Using the default FOV!
Loading input image...
Image shape: (3088, 2316, 3)
Running inference...
####### Please make sure the input image is in RGB format
Inference complete.
Parsing outputs...
Raw vertices shape: (18439, 3)
Raw faces shape: (36874, 3)
Creating mesh...
Exporting GLB...
Done.
Saved GLB: /workspace/3D body mesh preparation - 2/man_body_mesh.glb
Size: 664556 bytes


In [15]:
import trimesh

# Load the GLB file
mesh = trimesh.load('man_body_mesh.glb')

# Show the mesh
mesh.show()